# Module 03-02 — Hierarchical & Density-Based Clustering

**Learning objective:** Implement agglomerative hierarchical clustering (single, complete, and Ward linkage) and DBSCAN from scratch, visualise dendrograms, cut them at different heights, and understand when density-based methods succeed where K-Means fails.

**Prerequisites:** 03-01 (K-Means Clustering)

---

## Part 0 — Setup & Prerequisites

Module 03-01 showed that K-Means assumes convex, spherical clusters. This notebook introduces two paradigms that lift that restriction:

1. **Hierarchical clustering** — builds a tree of nested partitions. No need to specify $k$ in advance; the dendrogram reveals the cluster structure at every scale.
2. **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) — defines clusters as dense regions separated by low-density areas. Discovers arbitrary shapes and labels sparse points as *noise*.

We implement both from scratch and compare all three approaches (K-Means, hierarchical, DBSCAN) on challenging datasets where K-Means fails.

In [ ]:
import sys
import time
import copy
import random
import warnings
from typing import Optional
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.spatial.distance import cdist, squareform
from scipy.cluster.hierarchy import dendrogram as scipy_dendrogram
from scipy.cluster.hierarchy import linkage as scipy_linkage
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
)
from sklearn.cluster import (
    AgglomerativeClustering,
    DBSCAN as SklearnDBSCAN,
    KMeans as SklearnKMeans,
)
import torch
warnings.filterwarnings("ignore")

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy:  {np.__version__}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Configuration
N_SAMPLES  = 300   # samples for 2-D demonstration datasets
NOISE      = 0.07  # noise level for make_moons / make_circles
EPS        = 0.3   # DBSCAN neighbourhood radius
MIN_PTS    = 5     # DBSCAN minimum points for core classification

In [ ]:
# Generate datasets — all used for comparison later
X_blobs, y_blobs = make_blobs(
    n_samples=N_SAMPLES, centers=4, cluster_std=0.7, random_state=SEED
)
X_moons, y_moons = make_moons(n_samples=N_SAMPLES, noise=NOISE, random_state=SEED)
X_circles, y_circles = make_circles(
    n_samples=N_SAMPLES, noise=0.05, factor=0.5, random_state=SEED
)
# Anisotropic blobs
rng_tmp = np.random.RandomState(SEED)
X_aniso_raw, y_aniso = make_blobs(n_samples=N_SAMPLES, centers=3, random_state=SEED)
X_aniso = X_aniso_raw @ np.array([[0.6, -0.6], [-0.4, 0.8]])

datasets = {
    "Blobs":    (X_blobs,   y_blobs),
    "Moons":    (X_moons,   y_moons),
    "Circles":  (X_circles, y_circles),
    "Aniso":    (X_aniso,   y_aniso),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, (name, (X_d, y_d)) in zip(axes, datasets.items()):
    ax.scatter(X_d[:, 0], X_d[:, 1], c=y_d, cmap="tab10", s=12, alpha=0.7)
    ax.set_title(name)
    ax.axis("off")
plt.suptitle("Datasets (ground-truth colours)", fontsize=11)
plt.tight_layout()
plt.show()

---

## Part 1 — Agglomerative Hierarchical Clustering

### The Algorithm

Agglomerative clustering builds a cluster tree **bottom-up**:

1. Start: every point is its own cluster ($n$ clusters).
2. Repeat until one cluster remains:
   a. Find the two closest clusters according to the **linkage criterion**.
   b. Merge them into a single cluster.
3. The result is a **dendrogram** — a binary tree recording merge order and merge heights.

The choice of linkage determines cluster shapes:

| Linkage | Distance between clusters $A$, $B$ | Tends to produce |
|---------|-------------------------------------|-----------------|
| **Single** | $\min_{a \in A, b \in B} d(a, b)$ | Elongated "chaining" clusters |
| **Complete** | $\max_{a \in A, b \in B} d(a, b)$ | Compact, roughly spherical clusters |
| **Ward** | Increase in total WCSS when merging | Most balanced, compact clusters |

### Complexity

Naïve implementation: $O(n^3)$ — merge loop runs $n-1$ times, each requiring a distance matrix update of $O(n^2)$. The efficient SLINK algorithm achieves $O(n^2)$ for single linkage.

In [ ]:
def compute_pairwise_distances(X: np.ndarray) -> np.ndarray:
    """Compute the full pairwise Euclidean distance matrix.

    Args:
        X: Data matrix of shape (n_samples, n_features).

    Returns:
        Symmetric distance matrix of shape (n_samples, n_samples),
        with zeros on the diagonal.
    """
    return cdist(X, X, metric="euclidean")


def single_linkage(
    dist_a: np.ndarray,
    dist_b: np.ndarray,
) -> float:
    """Single linkage: minimum pairwise distance between clusters.

    Args:
        dist_a: Distances from cluster A members to all other points.
        dist_b: Distances from cluster B members to all other points.

    Returns:
        Minimum distance between any member of A and any member of B.
    """
    return float(np.min(np.minimum(dist_a, dist_b)))


def complete_linkage(
    dist_a: np.ndarray,
    dist_b: np.ndarray,
) -> float:
    """Complete linkage: maximum pairwise distance between clusters.

    Args:
        dist_a: Distances from cluster A members to all other points.
        dist_b: Distances from cluster B members to all other points.

    Returns:
        Maximum distance between any member of A and any member of B.
    """
    return float(np.max(np.maximum(dist_a, dist_b)))


def ward_linkage(
    X: np.ndarray,
    cluster_a: list,
    cluster_b: list,
) -> float:
    """Ward linkage: WCSS increase when merging two clusters.

    Ward's criterion minimises the total within-cluster variance.
    The merge cost equals the increase in total WCSS:
      delta = |A||B| / (|A|+|B|) * ||mu_A - mu_B||^2

    Args:
        X: Full data matrix of shape (n_samples, n_features).
        cluster_a: Indices of points in cluster A.
        cluster_b: Indices of points in cluster B.

    Returns:
        Ward merge cost (increase in total WCSS).
    """
    n_a, n_b = len(cluster_a), len(cluster_b)
    mu_a = X[cluster_a].mean(axis=0)
    mu_b = X[cluster_b].mean(axis=0)
    return float(n_a * n_b / (n_a + n_b) * np.sum((mu_a - mu_b) ** 2))

In [ ]:
def agglomerative_clustering(
    X: np.ndarray,
    linkage: str = "ward",
) -> list:
    """Agglomerative hierarchical clustering from scratch.

    Produces a merge history list compatible with scipy's dendrogram format.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        linkage: Linkage strategy — "single", "complete", or "ward".

    Returns:
        Linkage matrix of shape (n_samples-1, 4) where each row is
        [cluster_i, cluster_j, merge_distance, new_cluster_size].
        This matches scipy's linkage matrix format.
    """
    n = len(X)
    # Each cluster is a list of point indices
    clusters: list = [[i] for i in range(n)]
    # Track next cluster ID (new merged clusters get IDs >= n)
    next_id = n
    # Map cluster_id -> list of original point indices
    id_to_members: dict = {i: [i] for i in range(n)}
    # Active cluster IDs
    active_ids: list = list(range(n))

    dist_matrix = compute_pairwise_distances(X)
    linkage_matrix: list = []

    while len(active_ids) > 1:
        best_dist = float("inf")
        best_pair = (0, 1)

        for ii in range(len(active_ids)):
            for jj in range(ii + 1, len(active_ids)):
                id_i = active_ids[ii]
                id_j = active_ids[jj]
                members_i = id_to_members[id_i]
                members_j = id_to_members[id_j]

                if linkage == "single":
                    sub = dist_matrix[np.ix_(members_i, members_j)]
                    d = float(sub.min())
                elif linkage == "complete":
                    sub = dist_matrix[np.ix_(members_i, members_j)]
                    d = float(sub.max())
                elif linkage == "ward":
                    d = ward_linkage(X, members_i, members_j)
                else:
                    raise ValueError(f"Unknown linkage: {linkage!r}")

                if d < best_dist:
                    best_dist = d
                    best_pair = (id_i, id_j)

        id_i, id_j = best_pair
        members_i  = id_to_members[id_i]
        members_j  = id_to_members[id_j]
        new_members = members_i + members_j

        linkage_matrix.append([
            float(id_i),
            float(id_j),
            best_dist,
            float(len(new_members)),
        ])

        id_to_members[next_id] = new_members
        active_ids.remove(id_i)
        active_ids.remove(id_j)
        active_ids.append(next_id)
        next_id += 1

    return np.array(linkage_matrix)


def cut_dendrogram(
    linkage_matrix: np.ndarray,
    n_clusters: int,
    n_samples: int,
) -> np.ndarray:
    """Cut a linkage matrix to produce flat cluster labels.

    Simulates cutting the dendrogram at a height that yields exactly n_clusters
    by stopping the merge process early.

    Args:
        linkage_matrix: Linkage matrix from agglomerative_clustering,
                        shape (n_samples-1, 4).
        n_clusters: Desired number of clusters.
        n_samples: Number of original data points.

    Returns:
        Integer label array of shape (n_samples,).
    """
    # Replay the merges but stop when we have n_clusters remaining
    active: dict = {i: [i] for i in range(n_samples)}
    next_id = n_samples

    # Number of merges to perform
    n_merges = n_samples - n_clusters

    for row_idx in range(n_merges):
        id_i = int(linkage_matrix[row_idx, 0])
        id_j = int(linkage_matrix[row_idx, 1])
        new_members = active[id_i] + active[id_j]
        del active[id_i]
        del active[id_j]
        active[next_id] = new_members
        next_id += 1

    labels = np.zeros(n_samples, dtype=int)
    for cluster_label, members in enumerate(active.values()):
        for idx in members:
            labels[idx] = cluster_label

    return labels

### Sanity check — compare linkage methods on blobs

We run all three linkage strategies on the blobs dataset and compare their results.

In [ ]:
# Use a small subset for speed (O(n^3) scratch implementation)
N_SMALL = 80
rng_s = np.random.RandomState(SEED)
idx_s = rng_s.choice(len(X_blobs), size=N_SMALL, replace=False)
X_small, y_small = X_blobs[idx_s], y_blobs[idx_s]

print("Building linkage matrices (scratch, n=80)...")
linkage_results = {}
for lnk in ["single", "complete", "ward"]:
    t0 = time.perf_counter()
    Z = agglomerative_clustering(X_small, linkage=lnk)
    elapsed = time.perf_counter() - t0
    labels = cut_dendrogram(Z, n_clusters=4, n_samples=N_SMALL)
    ari = adjusted_rand_score(y_small, labels)
    linkage_results[lnk] = {"Z": Z, "labels": labels, "ari": ari, "time": elapsed}
    print(f"  {lnk:<10} | ARI={ari:.4f} | time={elapsed:.2f}s")

---

## Part 2 — Dendrograms & Cutting

### Reading a Dendrogram

A dendrogram plots the merge tree:
- **X-axis:** individual data points (leaves) and merged clusters.
- **Y-axis:** merge height (distance at which two clusters were joined).
- **Cutting horizontally** at a given height $h$ yields a flat partition: all clusters not yet merged at height $h$.

Choosing $h$ is equivalent to choosing $k$ after the fact — but the dendrogram lets you *see* the cluster structure before committing to any specific $k$.

In [ ]:
# Build linkage matrices using scipy for fast large-dataset dendrograms
print("Building scipy linkage matrices for full blobs dataset (n=300)...")
Z_single   = scipy_linkage(X_blobs, method="single",   metric="euclidean")
Z_complete = scipy_linkage(X_blobs, method="complete",  metric="euclidean")
Z_ward     = scipy_linkage(X_blobs, method="ward")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (title, Z) in zip(axes, [
    ("Single Linkage",   Z_single),
    ("Complete Linkage", Z_complete),
    ("Ward Linkage",     Z_ward),
]):
    scipy_dendrogram(Z, ax=ax, truncate_mode="lastp", p=20,
                     leaf_rotation=90, leaf_font_size=8, no_labels=True,
                     color_threshold=0.7 * max(Z[:, 2]))
    ax.set_title(title)
    ax.set_xlabel("Sample index (truncated)")
    ax.set_ylabel("Merge distance")

plt.suptitle("Dendrograms — Three Linkage Strategies on Blobs Dataset", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
def plot_dendrogram_with_cut(
    Z: np.ndarray,
    X: np.ndarray,
    cut_heights: list,
    title: str = "Dendrogram",
) -> None:
    """Plot a dendrogram alongside the resulting flat clustering at each cut height.

    Args:
        Z: Linkage matrix from scipy_linkage, shape (n-1, 4).
        X: Original data matrix (n_samples, 2) — used for scatter plots.
        cut_heights: List of heights at which to cut the dendrogram.
        title: Figure title.
    """
    from scipy.cluster.hierarchy import fcluster

    n_cuts = len(cut_heights)
    fig, axes = plt.subplots(1, n_cuts + 1, figsize=(4 * (n_cuts + 1), 4))

    # Dendrogram
    scipy_dendrogram(Z, ax=axes[0], truncate_mode="lastp", p=15,
                     leaf_rotation=90, leaf_font_size=7, no_labels=True)
    axes[0].set_title("Dendrogram")
    axes[0].set_ylabel("Merge distance")
    for h in cut_heights:
        axes[0].axhline(y=h, linestyle="--", linewidth=1, alpha=0.7)

    for ax, h in zip(axes[1:], cut_heights):
        labels_cut = fcluster(Z, t=h, criterion="distance") - 1
        k_cut = len(np.unique(labels_cut))
        ax.scatter(X[:, 0], X[:, 1], c=labels_cut, cmap="tab10", s=15, alpha=0.7)
        ax.set_title(f"Cut at h={h:.1f}  (k={k_cut})")
        ax.axis("off")

    plt.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()


# Show Ward linkage at different cut heights
plot_dendrogram_with_cut(
    Z_ward, X_blobs,
    cut_heights=[2.0, 5.0, 9.0],
    title="Ward Dendrogram — Different Cut Heights on Blobs",
)

### sklearn Validation

We compare our scratch agglomerative implementation against sklearn's `AgglomerativeClustering`.

In [ ]:
# Validate scratch implementation against sklearn (on small dataset)
print("Validation: scratch vs sklearn AgglomerativeClustering (n=80):")
for lnk in ["single", "complete", "ward"]:
    sk_agg = AgglomerativeClustering(n_clusters=4, linkage=lnk)
    sk_labels = sk_agg.fit_predict(X_small)
    scratch_labels = linkage_results[lnk]["labels"]

    scratch_ari = adjusted_rand_score(y_small, scratch_labels)
    sklearn_ari  = adjusted_rand_score(y_small, sk_labels)
    print(f"  {lnk:<10} | scratch ARI={scratch_ari:.4f}  sklearn ARI={sklearn_ari:.4f}  "
          f"diff={abs(scratch_ari - sklearn_ari):.4f}")

---

## Part 3 — DBSCAN from Scratch

### The Algorithm

DBSCAN classifies every point as one of three types based on its $\varepsilon$-neighbourhood (all points within distance $\varepsilon$):

| Type | Definition |
|------|-----------|
| **Core point** | Has $\geq \text{MinPts}$ points in its $\varepsilon$-neighbourhood (including itself) |
| **Border point** | Within $\varepsilon$ of a core point but not itself a core point |
| **Noise point** | Neither core nor border — isolated outlier |

**Cluster formation:** Two core points are in the same cluster if they are **density-reachable** — connected by a chain of core points each within $\varepsilon$ of the next. Border points are assigned to the cluster of their nearest core point.

### Key Parameters

- $\varepsilon$ (eps): neighbourhood radius — larger values merge more points.
- MinPts: minimum density threshold — higher values require denser cores, labels more points as noise.

### Complexity

$O(n^2)$ naïve (pairwise distances); $O(n \log n)$ with a spatial index (kd-tree / ball-tree).

In [ ]:
def dbscan(
    X: np.ndarray,
    eps: float = EPS,
    min_pts: int = MIN_PTS,
) -> np.ndarray:
    """DBSCAN density-based clustering from scratch.

    Classifies every point as core, border, or noise, then assigns cluster
    labels by expanding clusters from core points via BFS.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        eps: Neighbourhood radius epsilon.
        min_pts: Minimum points in eps-neighbourhood to be a core point.

    Returns:
        Integer label array of shape (n_samples,) where -1 denotes noise,
        and non-negative integers denote cluster membership.
    """
    n = len(X)
    # Pairwise distance matrix
    dist_matrix = cdist(X, X, metric="euclidean")

    # Classify each point
    UNVISITED  = -2
    NOISE      = -1

    labels = np.full(n, UNVISITED, dtype=int)
    # Precompute neighbourhoods
    neighbourhoods = [
        np.where(dist_matrix[i] <= eps)[0].tolist()
        for i in range(n)
    ]
    is_core = np.array([len(nbr) >= min_pts for nbr in neighbourhoods])

    cluster_id = 0

    for i in range(n):
        if labels[i] != UNVISITED:
            continue  # already processed

        if not is_core[i]:
            labels[i] = NOISE
            continue

        # Expand cluster from core point i using BFS
        labels[i] = cluster_id
        queue = deque(neighbourhoods[i])

        while queue:
            j = queue.popleft()
            if labels[j] == NOISE:
                labels[j] = cluster_id   # border point -> assign cluster
            if labels[j] != UNVISITED:
                continue

            labels[j] = cluster_id
            if is_core[j]:
                # Add j's neighbours to the queue
                for k in neighbourhoods[j]:
                    if labels[k] == UNVISITED or labels[k] == NOISE:
                        queue.append(k)

        cluster_id += 1

    # Any remaining UNVISITED (shouldn't happen) -> noise
    labels[labels == UNVISITED] = NOISE
    return labels


def classify_point_types(
    X: np.ndarray,
    labels: np.ndarray,
    eps: float,
    min_pts: int,
) -> tuple:
    """Return boolean masks for core, border, and noise points.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        labels: DBSCAN labels from dbscan(), shape (n_samples,).
        eps: Neighbourhood radius used in DBSCAN.
        min_pts: MinPts threshold used in DBSCAN.

    Returns:
        Tuple of (is_core, is_border, is_noise) boolean arrays.
    """
    dist_matrix = cdist(X, X, metric="euclidean")
    neighbour_counts = (dist_matrix <= eps).sum(axis=1)
    is_core   = neighbour_counts >= min_pts
    is_noise  = labels == -1
    is_border = (~is_core) & (~is_noise)
    return is_core, is_border, is_noise

### Sanity check — DBSCAN on moons dataset

In [ ]:
t0 = time.perf_counter()
labels_dbscan_moons = dbscan(X_moons, eps=EPS, min_pts=MIN_PTS)
elapsed_dbscan = time.perf_counter() - t0

is_core, is_border, is_noise = classify_point_types(
    X_moons, labels_dbscan_moons, eps=EPS, min_pts=MIN_PTS
)

n_clusters_found = len(set(labels_dbscan_moons)) - (1 if -1 in labels_dbscan_moons else 0)
print(f"DBSCAN on moons (eps={EPS}, min_pts={MIN_PTS}):")
print(f"  Clusters found : {n_clusters_found}")
print(f"  Core points    : {is_core.sum()}")
print(f"  Border points  : {is_border.sum()}")
print(f"  Noise points   : {is_noise.sum()}")
print(f"  Time           : {elapsed_dbscan:.3f}s")
print(f"  ARI vs truth   : {adjusted_rand_score(y_moons, labels_dbscan_moons):.4f}")

# Visualise point types
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: cluster labels
scatter = axes[0].scatter(X_moons[:, 0], X_moons[:, 1],
                          c=labels_dbscan_moons, cmap="tab10",
                          s=20, alpha=0.8)
axes[0].set_title("DBSCAN — cluster labels (-1 = noise)")
axes[0].set_xlabel("Feature 1")
axes[0].set_ylabel("Feature 2")

# Right: point types
type_colors = np.where(is_core, 0, np.where(is_border, 1, 2))
cmap_types = plt.cm.get_cmap("Set1", 3)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1],
                c=type_colors, cmap=cmap_types, s=20, alpha=0.8)
legend_els = [
    mpatches.Patch(color=cmap_types(0), label=f"Core ({is_core.sum()})"),
    mpatches.Patch(color=cmap_types(1), label=f"Border ({is_border.sum()})"),
    mpatches.Patch(color=cmap_types(2), label=f"Noise ({is_noise.sum()})"),
]
axes[1].legend(handles=legend_els, loc="upper right")
axes[1].set_title("DBSCAN — point type classification")

plt.tight_layout()
plt.show()

---

## Part 4 — Parameter Sensitivity & sklearn Validation

### 4.1 DBSCAN Parameter Sweep

Both $\varepsilon$ and MinPts dramatically affect the number of clusters and noise fraction. We sweep both parameters and visualise their effect.

In [ ]:
def dbscan_param_sweep(
    X: np.ndarray,
    eps_values: list,
    min_pts_values: list,
    y_true: np.ndarray | None = None,
) -> pd.DataFrame:
    """Sweep eps and min_pts for DBSCAN and record quality metrics.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        eps_values: List of epsilon values to test.
        min_pts_values: List of MinPts values to test.
        y_true: Optional ground truth for ARI computation.

    Returns:
        DataFrame with columns [eps, min_pts, n_clusters, noise_pct,
                                 silhouette, ari].
    """
    rows = []
    for eps in eps_values:
        for min_pts in min_pts_values:
            labels = dbscan(X, eps=eps, min_pts=min_pts)
            unique = set(labels)
            n_clusters = len(unique) - (1 if -1 in unique else 0)
            noise_pct  = float((labels == -1).mean() * 100)

            if n_clusters >= 2 and (labels != -1).sum() >= 2:
                # Silhouette only on non-noise points
                non_noise = labels != -1
                sil = silhouette_score(X[non_noise], labels[non_noise])
            else:
                sil = float("nan")

            ari = float("nan")
            if y_true is not None:
                ari = adjusted_rand_score(y_true, labels)

            rows.append({
                "eps": eps, "min_pts": min_pts,
                "n_clusters": n_clusters,
                "noise_pct": noise_pct,
                "silhouette": sil,
                "ari": ari,
            })

    return pd.DataFrame(rows)


eps_sweep     = [0.1, 0.2, 0.3, 0.4, 0.5]
minpts_sweep  = [3, 5, 8]

print("DBSCAN parameter sweep on moons dataset:")
sweep_df = dbscan_param_sweep(
    X_moons, eps_sweep, minpts_sweep, y_true=y_moons
)
print(sweep_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

In [ ]:
# Heatmap of n_clusters over eps x min_pts grid
pivot_nclusters = sweep_df.pivot(index="min_pts", columns="eps", values="n_clusters")
pivot_noise     = sweep_df.pivot(index="min_pts", columns="eps", values="noise_pct")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, pivot, title, fmt in [
    (axes[0], pivot_nclusters, "Number of Clusters", ".0f"),
    (axes[1], pivot_noise,     "Noise Fraction (%)", ".1f"),
]:
    im = ax.imshow(pivot.values, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{v:.1f}" for v in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index.tolist())
    ax.set_xlabel("eps")
    ax.set_ylabel("min_pts")
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
    for r in range(pivot.shape[0]):
        for c in range(pivot.shape[1]):
            val = pivot.values[r, c]
            ax.text(c, r, format(val, fmt), ha="center", va="center",
                    fontsize=9, color="white" if val > pivot.values.max() / 2 else "black")

plt.tight_layout()
plt.show()

### 4.2 sklearn Validation — DBSCAN

In [ ]:
# sklearn DBSCAN validation
sk_dbscan = SklearnDBSCAN(eps=EPS, min_samples=MIN_PTS)
sk_labels_moons = sk_dbscan.fit_predict(X_moons)

scratch_ari = adjusted_rand_score(y_moons, labels_dbscan_moons)
sklearn_ari  = adjusted_rand_score(y_moons, sk_labels_moons)

n_scratch = len(set(labels_dbscan_moons)) - (1 if -1 in labels_dbscan_moons else 0)
n_sklearn  = len(set(sk_labels_moons))    - (1 if -1 in sk_labels_moons    else 0)

print("DBSCAN validation — scratch vs sklearn (moons dataset):")
print(f"  Scratch: {n_scratch} clusters  ARI={scratch_ari:.4f}  "
      f"noise={int((labels_dbscan_moons == -1).sum())}")
print(f"  sklearn: {n_sklearn} clusters  ARI={sklearn_ari:.4f}  "
      f"noise={int((sk_labels_moons == -1).sum())}")
print(f"  Agreement: {float(np.mean(labels_dbscan_moons == sk_labels_moons)):.4f}")

---

## Part 4 continued — Head-to-Head Comparison

### 4.3 All Clustering Algorithms on All Datasets

We evaluate K-Means, Ward hierarchical clustering, and DBSCAN on all four datasets to show where each method wins and where it fails.

> **Note:** K-Means is reviewed here — see 03-01 for full derivation.

In [ ]:
def run_all_algorithms(
    X: np.ndarray,
    y_true: np.ndarray,
    k_true: int,
    eps: float = EPS,
    min_pts: int = MIN_PTS,
) -> dict:
    """Run K-Means, Ward, and DBSCAN on a dataset and return quality metrics.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        y_true: Ground truth labels for ARI computation.
        k_true: True number of clusters (used for K-Means and Ward).
        eps: DBSCAN epsilon.
        min_pts: DBSCAN min_pts.

    Returns:
        Dict mapping algorithm name to dict of {labels, ari, n_clusters, noise_pct}.
    """
    results: dict = {}

    # K-Means (sklearn for speed)
    km = SklearnKMeans(n_clusters=k_true, init="k-means++",
                       n_init=10, random_state=SEED)
    km_labels = km.fit_predict(X)
    results["K-Means"] = {
        "labels": km_labels,
        "ari": adjusted_rand_score(y_true, km_labels),
        "n_clusters": k_true,
        "noise_pct": 0.0,
    }

    # Ward hierarchical (sklearn for speed)
    ward = AgglomerativeClustering(n_clusters=k_true, linkage="ward")
    ward_labels = ward.fit_predict(X)
    results["Ward"] = {
        "labels": ward_labels,
        "ari": adjusted_rand_score(y_true, ward_labels),
        "n_clusters": k_true,
        "noise_pct": 0.0,
    }

    # DBSCAN
    db_labels = dbscan(X, eps=eps, min_pts=min_pts)
    n_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    results["DBSCAN"] = {
        "labels": db_labels,
        "ari": adjusted_rand_score(y_true, db_labels),
        "n_clusters": n_db,
        "noise_pct": float((db_labels == -1).mean() * 100),
    }

    return results


# Dataset-specific DBSCAN parameters
dbscan_params = {
    "Blobs":   (0.7, 5),
    "Moons":   (0.3, 5),
    "Circles": (0.2, 5),
    "Aniso":   (0.7, 5),
}
k_true_map = {
    "Blobs": 4, "Moons": 2, "Circles": 2, "Aniso": 3
}

all_results = {}
for ds_name, (X_d, y_d) in datasets.items():
    eps_d, mpts_d = dbscan_params[ds_name]
    all_results[ds_name] = run_all_algorithms(
        X_d, y_d, k_true=k_true_map[ds_name], eps=eps_d, min_pts=mpts_d
    )
print("All algorithms evaluated on all datasets.")

In [ ]:
# Visualise the 4x3 grid of results
fig, axes = plt.subplots(4, 3, figsize=(14, 16))
algorithm_names = ["K-Means", "Ward", "DBSCAN"]

for row_idx, (ds_name, (X_d, y_d)) in enumerate(datasets.items()):
    for col_idx, alg_name in enumerate(algorithm_names):
        ax = axes[row_idx, col_idx]
        res = all_results[ds_name][alg_name]
        labels = res["labels"]
        ari    = res["ari"]
        n_cl   = res["n_clusters"]
        noise  = res["noise_pct"]

        scatter = ax.scatter(
            X_d[:, 0], X_d[:, 1],
            c=labels, cmap="tab10", s=12, alpha=0.8,
        )
        noise_title = f"  noise={noise:.0f}%" if noise > 0 else ""
        ax.set_title(
            f"{alg_name} on {ds_name}\nARI={ari:.3f}  k={n_cl}{noise_title}",
            fontsize=8,
        )
        ax.axis("off")

plt.suptitle(
    "K-Means vs Ward vs DBSCAN — All Datasets",
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table: ARI by algorithm and dataset
print("ARI Summary (higher = better, max 1.0):")
print("-" * 45)
header = f"{'Dataset':<12}" + "".join(f"{a:<12}" for a in algorithm_names)
print(header)
print("-" * 45)
for ds_name in datasets:
    row = f"{ds_name:<12}"
    for alg_name in algorithm_names:
        ari = all_results[ds_name][alg_name]["ari"]
        row += f"{ari:<12.4f}"
    print(row)

print("\nKey insight:")
print("  K-Means succeeds on Blobs but fails on Moons and Circles.")
print("  DBSCAN handles non-convex shapes but requires careful eps/min_pts tuning.")
print("  Ward is a good general-purpose choice when k is known.")

### 4.4 Linkage Comparison — Single vs Complete vs Ward

Each linkage strategy is suited for different cluster geometries. We visualise all three on challenging datasets to build intuition.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
linkages = ["single", "complete", "ward"]
ds_list  = list(datasets.keys())

for row_idx, lnk in enumerate(linkages):
    for col_idx, ds_name in enumerate(ds_list):
        ax = axes[row_idx, col_idx]
        X_d, y_d = datasets[ds_name]
        k_d = k_true_map[ds_name]
        agg = AgglomerativeClustering(n_clusters=k_d, linkage=lnk)
        labels_lnk = agg.fit_predict(X_d)
        ari_lnk = adjusted_rand_score(y_d, labels_lnk)
        ax.scatter(X_d[:, 0], X_d[:, 1], c=labels_lnk,
                   cmap="tab10", s=12, alpha=0.8)
        ax.set_title(f"{lnk.capitalize()} on {ds_name}\nARI={ari_lnk:.3f}",
                     fontsize=8)
        ax.axis("off")

plt.suptitle("Linkage Strategy Comparison — All Datasets", fontsize=12)
plt.tight_layout()
plt.show()
print("Single linkage suffers from chaining; complete is more compact;")
print("Ward balances cluster size and cohesion well on Gaussian blobs.")

### 4.5 DBSCAN Noise Robustness

DBSCAN explicitly labels outliers as noise. We test how well it isolates injected outliers from genuine cluster members.

In [ ]:
def inject_outliers(
    X: np.ndarray,
    y: np.ndarray,
    n_outliers: int,
    scale: float = 4.0,
    seed: int = SEED,
) -> tuple:
    """Inject uniformly-distributed outliers into a dataset.

    Args:
        X: Original data matrix.
        y: Original labels.
        n_outliers: Number of outlier points to add.
        scale: Range of outlier coordinates (uniform in [-scale, scale]).
        seed: Random seed.

    Returns:
        Tuple of (augmented_X, augmented_y) where outliers have label -1.
    """
    rng = np.random.RandomState(seed)
    outliers = rng.uniform(-scale, scale, size=(n_outliers, X.shape[1]))
    X_aug = np.vstack([X, outliers])
    y_aug = np.concatenate([y, np.full(n_outliers, -1, dtype=int)])
    return X_aug, y_aug


X_noisy, y_noisy = inject_outliers(X_moons, y_moons, n_outliers=20)

# K-Means vs DBSCAN on outlier-contaminated data
km_noisy = SklearnKMeans(n_clusters=2, init="k-means++",
                         n_init=10, random_state=SEED)
km_noisy_labels = km_noisy.fit_predict(X_noisy)
db_noisy_labels = dbscan(X_noisy, eps=EPS, min_pts=MIN_PTS)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (labels_plot, title_plot) in zip(axes, [
    (y_noisy,        "Ground truth (grey = outliers)"),
    (km_noisy_labels, "K-Means — outliers distort centroids"),
    (db_noisy_labels, "DBSCAN — outliers labelled as noise (-1)"),
]):
    ax.scatter(X_noisy[:, 0], X_noisy[:, 1], c=labels_plot,
               cmap="tab10", s=15, alpha=0.7)
    ax.set_title(title_plot, fontsize=9)
    ax.axis("off")

plt.suptitle("Outlier Robustness: K-Means vs DBSCAN", fontsize=11)
plt.tight_layout()
plt.show()

# Measure how many injected outliers were correctly labelled as noise by DBSCAN
n_total = len(X_noisy)
n_clean = len(X_moons)
injected_as_noise = (db_noisy_labels[n_clean:] == -1).mean()
clean_as_noise    = (db_noisy_labels[:n_clean] == -1).mean()
print(f"Injected outliers correctly labelled as noise: {injected_as_noise:.1%}")
print(f"Clean points incorrectly labelled as noise   : {clean_as_noise:.1%}")

### 4.6 Hierarchical Clustering on Digits (high-dimensional)

We apply Ward hierarchical clustering to the 64-D Digits dataset to show it works beyond 2-D toy examples.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

digits = load_digits()
X_dig, y_dig = digits.data, digits.target

# Reduce to 30 PCA components first (speeds up distance computations)
# See 03-03 for full PCA treatment
pca_pre = PCA(n_components=30, random_state=SEED)
X_dig_pca = pca_pre.fit_transform(X_dig)

# Ward clustering with k = 10 (one per digit)
ward_dig = AgglomerativeClustering(n_clusters=10, linkage="ward")
labels_ward_dig = ward_dig.fit_predict(X_dig_pca)

ari_ward_dig = adjusted_rand_score(y_dig, labels_ward_dig)
sil_ward_dig = silhouette_score(X_dig_pca, labels_ward_dig)
print(f"Ward clustering on Digits (PCA-30D, k=10):")
print(f"  ARI       : {ari_ward_dig:.4f}")
print(f"  Silhouette: {sil_ward_dig:.4f}")

# Visualise in 2-D PCA space
pca_2d = PCA(n_components=2, random_state=SEED)
X_dig_2d = pca_2d.fit_transform(X_dig_pca)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_dig_2d[:, 0], X_dig_2d[:, 1], c=y_dig,
                cmap="tab10", s=8, alpha=0.6)
axes[0].set_title("Ground truth labels (2-D PCA projection)")
axes[0].axis("off")
axes[1].scatter(X_dig_2d[:, 0], X_dig_2d[:, 1], c=labels_ward_dig,
                cmap="tab10", s=8, alpha=0.6)
axes[1].set_title(f"Ward clustering k=10  ARI={ari_ward_dig:.3f}")
axes[1].axis("off")
plt.tight_layout()
plt.show()
print("(PCA used here for speed only — see 03-03 for full PCA treatment.)")

### 4.7 Cophenetic Correlation Coefficient

The **cophenetic correlation coefficient** measures how faithfully a dendrogram preserves the original pairwise distances. For each pair $(i, j)$, the cophenetic distance is the height at which $i$ and $j$ first merge in the tree. A high correlation (close to 1.0) means the dendrogram is a faithful representation of the true inter-point distances.

$$r_c = \frac{\sum_{i < j}(d_{ij} - \bar{d})(c_{ij} - \bar{c})}
             {\sqrt{\sum_{i < j}(d_{ij} - \bar{d})^2 \cdot \sum_{i < j}(c_{ij} - \bar{c})^2}}$$

where $d_{ij}$ = original distance, $c_{ij}$ = cophenetic distance.

In [ ]:
def cophenetic_correlation(
    dist_vector: np.ndarray,
    Z: np.ndarray,
    n_samples: int,
) -> float:
    """Compute cophenetic correlation between original distances and dendrogram heights.

    Args:
        dist_vector: Condensed distance vector from scipy.spatial.distance.pdist,
                     shape (n*(n-1)//2,).
        Z: Linkage matrix from scipy_linkage, shape (n-1, 4).
        n_samples: Number of original data points.

    Returns:
        Pearson correlation between original and cophenetic distances.
    """
    from scipy.spatial.distance import pdist
    from scipy.cluster.hierarchy import cophenet

    coph_dists, coph_corr = cophenet(Z, dist_vector)
    return float(coph_corr)


def compute_cophenetic_for_linkages(
    X: np.ndarray,
    linkages: list,
) -> dict:
    """Compute cophenetic correlation for multiple linkage strategies.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        linkages: List of linkage strategy names to test.

    Returns:
        Dict mapping linkage name to cophenetic correlation.
    """
    from scipy.spatial.distance import pdist
    from scipy.cluster.hierarchy import cophenet

    dist_vec = pdist(X, metric="euclidean")
    results: dict = {}
    for lnk in linkages:
        Z = scipy_linkage(X, method=lnk)
        _, cc = cophenet(Z, dist_vec)
        results[lnk] = float(cc)
        print(f"  {lnk:<10}: cophenetic r = {cc:.4f}")
    return results


print("Cophenetic correlation (higher = dendrogram preserves distances better):")
coph_results = compute_cophenetic_for_linkages(
    X_blobs, linkages=["single", "complete", "ward", "average"]
)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(list(coph_results.keys()), list(coph_results.values()),
       color=["#4472C4", "#70AD47", "#ED7D31", "#9E480E"],
       edgecolor="black", linewidth=0.6)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Cophenetic Correlation")
ax.set_title("Cophenetic Correlation by Linkage Strategy — Blobs")
for i, (name, val) in enumerate(coph_results.items()):
    ax.text(i, val + 0.02, f"{val:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()
print("Higher cophenetic correlation means the dendrogram is a better summary of the data.")

### 4.8 Automatic k Selection from Dendrogram

Instead of manually inspecting the dendrogram, we can find the largest *gap* between consecutive merge heights in the Ward linkage matrix. The $k$ corresponding to this gap is a data-driven estimate of the optimal number of clusters.

In [ ]:
def select_k_from_dendrogram(
    Z: np.ndarray,
    n_samples: int,
) -> tuple:
    """Select k by finding the largest merge-height gap in the dendrogram.

    The merge history of a Ward dendrogram encodes cluster structure: a large
    jump in merge height suggests merging qualitatively different clusters.
    The optimal k is found at the step just before this big jump.

    Args:
        Z: Linkage matrix from scipy_linkage, shape (n-1, 4).
        n_samples: Number of original data points.

    Returns:
        Tuple of (optimal_k, last_k_merge_heights, all_gaps).
    """
    # Last few merge heights (when only a few clusters remain)
    merge_heights = Z[:, 2]
    # Consider only the final n_samples - 1 merges that reduce to small k
    # Look at the last 15 merges
    last_heights = merge_heights[-(min(15, n_samples - 1)):]
    gaps = np.diff(last_heights)
    # Largest gap index in the last_heights array
    max_gap_idx = int(np.argmax(gaps))
    # k = number of merges left after the gap
    k_opt = len(last_heights) - max_gap_idx - 1
    k_opt = max(2, k_opt)   # at least 2 clusters
    return k_opt, last_heights, gaps


k_opt_blobs, last_h, gaps_blobs = select_k_from_dendrogram(Z_ward, len(X_blobs))
print(f"Auto-selected k from Ward dendrogram: k = {k_opt_blobs}  (true k = 4)")

# Visualise the gap approach
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x_range = range(len(last_h))
axes[0].bar(x_range, last_h, color="#4472C4", edgecolor="black", linewidth=0.5)
axes[0].set_xlabel("Merge step (last 15)")
axes[0].set_ylabel("Merge height")
axes[0].set_title("Last 15 merge heights — Ward (Blobs)")

axes[1].bar(range(len(gaps_blobs)), gaps_blobs, color="#ED7D31",
            edgecolor="black", linewidth=0.5)
axes[1].axvline(x=int(np.argmax(gaps_blobs)), color="red", linestyle="--",
                label=f"Max gap -> k={k_opt_blobs}")
axes[1].set_xlabel("Gap index")
axes[1].set_ylabel("Height gap")
axes[1].set_title("Height gaps — auto-select k")
axes[1].legend()
plt.tight_layout()
plt.show()

# Apply auto-selected k
from scipy.cluster.hierarchy import fcluster
labels_auto = fcluster(Z_ward, t=k_opt_blobs, criterion="maxclust") - 1
ari_auto = adjusted_rand_score(y_blobs, labels_auto)
print(f"ARI with auto-selected k={k_opt_blobs}: {ari_auto:.4f}")

### 4.9 Distance Metric Comparison for Hierarchical Clustering

Euclidean distance is the default but not always optimal. Manhattan distance is more robust to high-dimensional data; cosine distance suits text or angular data. We compare three metrics on the blobs dataset.

In [ ]:
def hierarchical_metric_comparison(
    X: np.ndarray,
    y_true: np.ndarray,
    k: int,
    metrics: list,
    linkage: str = "complete",
) -> pd.DataFrame:
    """Compare hierarchical clustering under different distance metrics.

    Ward linkage is only compatible with Euclidean distance, so we use
    complete linkage for a fair metric comparison.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        y_true: Ground truth labels.
        k: Number of clusters to cut the dendrogram to.
        metrics: List of distance metric names (scipy-compatible).
        linkage: Linkage strategy (default "complete" — works with all metrics).

    Returns:
        DataFrame with columns [Metric, ARI, Silhouette, Time (s)].
    """
    rows = []
    for metric in metrics:
        t0 = time.perf_counter()
        Z_m = scipy_linkage(X, method=linkage, metric=metric)
        labels_m = fcluster(Z_m, t=k, criterion="maxclust") - 1
        elapsed = time.perf_counter() - t0
        ari_m = adjusted_rand_score(y_true, labels_m)
        sil_m = silhouette_score(X, labels_m)
        rows.append({
            "Metric": metric,
            "ARI": ari_m,
            "Silhouette": sil_m,
            "Time (s)": elapsed,
        })
        print(f"  {metric:<12} ARI={ari_m:.4f}  sil={sil_m:.4f}  t={elapsed:.3f}s")
    return pd.DataFrame(rows)


metrics_to_test = ["euclidean", "cityblock", "cosine", "chebyshev"]
print(f"Distance metric comparison (complete linkage, k=4, blobs):")
metric_df = hierarchical_metric_comparison(X_blobs, y_blobs, k=4, metrics=metrics_to_test)
print()
print(metric_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

### 4.10 DBSCAN on Higher-Dimensional Data

DBSCAN operates on any distance metric and any dimensionality. We test on a 3-D synthetic dataset to confirm the scratch implementation generalises beyond 2-D.

In [ ]:
def generate_3d_blobs_with_outliers(
    n_per_cluster: int = 80,
    n_outliers: int = 15,
    seed: int = SEED,
) -> tuple:
    """Generate a 3-D dataset with three Gaussian blobs and random outliers.

    Args:
        n_per_cluster: Points per blob.
        n_outliers: Number of uniformly distributed outlier points.
        seed: Random seed.

    Returns:
        Tuple of (X array shape (n_total, 3), y_true labels).
    """
    rng = np.random.RandomState(seed)
    centers = np.array([[0, 0, 0], [4, 0, 0], [2, 4, 0]], dtype=float)
    X_list, y_list = [], []
    for label, center in enumerate(centers):
        pts = rng.randn(n_per_cluster, 3) * 0.7 + center
        X_list.append(pts)
        y_list.append(np.full(n_per_cluster, label))
    # Outliers
    out = rng.uniform(-2, 6, size=(n_outliers, 3))
    X_list.append(out)
    y_list.append(np.full(n_outliers, -1))
    return np.vstack(X_list), np.concatenate(y_list)


X_3d, y_3d = generate_3d_blobs_with_outliers()
labels_3d  = dbscan(X_3d, eps=1.2, min_pts=5)

n_cl_3d = len(set(labels_3d)) - (1 if -1 in labels_3d else 0)
ari_3d   = adjusted_rand_score(y_3d[y_3d != -1], labels_3d[y_3d != -1])
noise_3d = (labels_3d == -1).sum()
print(f"3-D DBSCAN result: {n_cl_3d} clusters  noise={noise_3d}  ARI={ari_3d:.4f}")

# 3-D scatter plot
fig = plt.figure(figsize=(12, 4))
for subplot_idx, (labels_plot, title_plot) in enumerate([
    (y_3d,      "Ground truth"),
    (labels_3d, f"DBSCAN (k={n_cl_3d}, noise={noise_3d})"),
], start=1):
    ax = fig.add_subplot(1, 2, subplot_idx, projection="3d")
    sc = ax.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2],
                    c=labels_plot, cmap="tab10", s=12, alpha=0.7)
    ax.set_title(title_plot)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")

plt.suptitle("DBSCAN on 3-D Data with Outliers", fontsize=11)
plt.tight_layout()
plt.show()

### 4.11 Merge Cost Analysis — Understanding the Linkage Tree

For Ward linkage, the merge cost represents the increase in total within-cluster variance. By plotting merge costs in order we can directly see where "natural" cluster boundaries are.

In [ ]:
def analyse_merge_costs(
    Z: np.ndarray,
    n_show: int = 20,
) -> np.ndarray:
    """Extract and analyse the last n_show merge costs from a linkage matrix.

    Large merge costs signal natural cluster boundaries — merging qualitatively
    different groups incurs a high cost.

    Args:
        Z: Linkage matrix of shape (n-1, 4).
        n_show: Number of final merges to analyse.

    Returns:
        Array of the last n_show merge heights.
    """
    heights = Z[-n_show:, 2]
    return heights


ward_costs = analyse_merge_costs(Z_ward, n_show=15)
single_costs = analyse_merge_costs(
    scipy_linkage(X_blobs, method="single"), n_show=15
)
complete_costs = analyse_merge_costs(
    scipy_linkage(X_blobs, method="complete"), n_show=15
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (costs, title) in zip(axes, [
    (ward_costs,     "Ward merge costs (last 15)"),
    (single_costs,   "Single linkage heights (last 15)"),
    (complete_costs, "Complete linkage heights (last 15)"),
]):
    steps = list(range(len(costs)))
    ax.bar(steps, costs, color="#4472C4", edgecolor="black", linewidth=0.5)
    # Highlight the largest gap
    gaps_c = np.diff(costs)
    if len(gaps_c) > 0:
        max_g = int(np.argmax(gaps_c))
        ax.axvspan(max_g, max_g + 1, color="red", alpha=0.25,
                   label=f"Largest gap at step {max_g}")
        ax.legend(fontsize=7)
    ax.set_xlabel("Merge step (from end)")
    ax.set_ylabel("Merge height / cost")
    ax.set_title(title)

plt.suptitle("Merge Cost Analysis — Natural Cluster Boundaries", fontsize=11)
plt.tight_layout()
plt.show()
print("Large Ward merge costs correspond to merging naturally separate clusters.")
print("The gap BEFORE the highest bar shows where to cut for the best k.")

### 4.12 k-Selection Stability Across Datasets

We run automated k-selection (dendrogram gap, elbow-style silhouette peak) on all four datasets to evaluate how reliably these heuristics recover the true k.

In [ ]:
def auto_select_k_silhouette(
    X: np.ndarray,
    k_range: list,
    linkage: str = "ward",
) -> int:
    """Select k by maximising silhouette score over a range.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        k_range: List of k values to evaluate.
        linkage: Hierarchical linkage strategy.

    Returns:
        k value that maximises the mean silhouette score.
    """
    best_k   = k_range[0]
    best_sil = -1.0
    for k in k_range:
        if k < 2:
            continue
        agg = AgglomerativeClustering(n_clusters=k, linkage=linkage)
        labels_k = agg.fit_predict(X)
        if len(np.unique(labels_k)) < 2:
            continue
        sil = silhouette_score(X, labels_k)
        if sil > best_sil:
            best_sil = sil
            best_k   = k
    return best_k


def auto_select_k_gap(Z: np.ndarray, n_samples: int) -> int:
    """Select k from dendrogram using the largest merge-height gap.

    Args:
        Z: Linkage matrix from scipy_linkage.
        n_samples: Number of original data points.

    Returns:
        Estimated optimal k.
    """
    last_heights = Z[:, 2][-(min(12, n_samples - 1)):]
    gaps = np.diff(last_heights)
    if len(gaps) == 0:
        return 2
    max_gap_idx = int(np.argmax(gaps))
    k_est = len(last_heights) - max_gap_idx - 1
    return max(2, k_est)


print("Automatic k-selection stability across datasets:")
print("-" * 60)
print(f"{'Dataset':<12} {'True k':<8} {'Sil peak k':<12} {'Gap k':<10}")
print("-" * 60)

for ds_name, (X_d, y_d) in datasets.items():
    k_true_d = k_true_map[ds_name]
    k_sil = auto_select_k_silhouette(X_d, k_range=list(range(2, 8)))
    Z_d   = scipy_linkage(X_d, method="ward")
    k_gap = auto_select_k_gap(Z_d, len(X_d))
    correct_sil = "OK" if k_sil == k_true_d else f"got {k_sil}"
    correct_gap = "OK" if k_gap == k_true_d else f"got {k_gap}"
    print(f"{ds_name:<12} {k_true_d:<8} {correct_sil:<12} {correct_gap:<10}")

print("\nNote: Both heuristics struggle on Moons and Circles (non-Gaussian shapes).")
print("Automatic k-selection works best when clusters are well-separated Gaussians.")

### 4.13 Comprehensive Algorithm Comparison Table

A single reference table summarising all three algorithms across key dimensions.

In [ ]:
def build_algorithm_comparison_table() -> pd.DataFrame:
    """Build a reference table comparing K-Means, hierarchical, and DBSCAN.

    Returns:
        DataFrame with algorithm properties as rows.
    """
    rows = [
        {
            "Algorithm":         "K-Means",
            "Complexity":        "O(n k d T)",
            "Requires k":        "Yes",
            "Cluster shapes":    "Convex / spherical",
            "Handles noise":     "No",
            "Dendrogram":        "No",
            "Scalability":       "High (mini-batch)",
            "Reference":         "03-01",
        },
        {
            "Algorithm":         "Hierarchical (Ward)",
            "Complexity":        "O(n^2) - O(n^3)",
            "Requires k":        "No (cut after)",
            "Cluster shapes":    "Compact / Gaussian",
            "Handles noise":     "No",
            "Dendrogram":        "Yes",
            "Scalability":       "Low (n < 10k)",
            "Reference":         "03-02",
        },
        {
            "Algorithm":         "Hierarchical (Single)",
            "Complexity":        "O(n^2)",
            "Requires k":        "No (cut after)",
            "Cluster shapes":    "Elongated / chained",
            "Handles noise":     "Sensitive",
            "Dendrogram":        "Yes",
            "Scalability":       "Medium",
            "Reference":         "03-02",
        },
        {
            "Algorithm":         "DBSCAN",
            "Complexity":        "O(n^2) naive / O(n log n) indexed",
            "Requires k":        "No",
            "Cluster shapes":    "Arbitrary (non-convex)",
            "Handles noise":     "Yes (explicit)",
            "Dendrogram":        "No",
            "Scalability":       "Medium",
            "Reference":         "03-02",
        },
    ]
    return pd.DataFrame(rows)


comparison_table = build_algorithm_comparison_table()
print("Clustering Algorithm Reference Table:")
print()
print(comparison_table.to_string(index=False))

---

## Part 5 — Summary & Lessons Learned

### Key Takeaways

- **Agglomerative clustering** builds a cluster tree bottom-up by iteratively merging the closest pair of clusters. The merge history is captured in a **dendrogram** that lets you choose $k$ after the fact by cutting at any height.
- **Linkage choice matters:** single linkage suffers from "chaining" (merges elongated shapes); complete linkage produces compact, roughly spherical clusters; Ward minimises within-cluster variance and is usually the best all-around choice.
- **DBSCAN** defines clusters as dense regions and classifies every point as *core* (dense neighbourhood), *border* (reachable from core), or *noise* (isolated). It discovers arbitrarily shaped clusters and is robust to outliers — but requires careful tuning of $\varepsilon$ and MinPts.
- **When to use each method:**
  - K-Means: fast, scalable, clusters are convex and roughly equal in size.
  - Hierarchical: when you want multi-scale structure or don't know $k$; slower at $O(n^3)$ naïve.
  - DBSCAN: non-convex clusters, unknown $k$, outlier-contaminated data.
- **No free lunch:** all three algorithms fail on at least one of our four test datasets — choosing the right algorithm requires understanding the data geometry.

### What's Next

→ **03-03 (Principal Component Analysis)** reduces dimensionality before clustering, making high-dimensional data amenable to all methods introduced in this notebook.